# 🌲 Step 0: Smart Upload & Preview (Hour 1 Quickstart)

Upload your NFI Ground Truth data (CSV/Excel) or Study Area (GeoJSON).

**No data at hand?** Use the bundled sample: `data/sample_nfi.csv`.

> Running in Colab? Clone the repo first, then `%cd forest-carbon-continuity-pipeline/notebooks` so the relative `../src` paths resolve.

In [ ]:
import io
import os

import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, clear_output, display

print("🌲 STEP 1: Upload National Forest Inventory (NFI) Ground Truth Data")
print("Accepted Formats: .csv, .xlsx, .gpkg, .geojson   |   Sample: ../data/sample_nfi.csv")

nfi_uploader = widgets.FileUpload(accept='.csv,.xlsx,.xls,.gpkg,.geojson',
                                  multiple=False, description='Upload NFI Data')
display(nfi_uploader)


def on_upload_change(change):
    if not change['new']:
        return
    clear_output(wait=True)
    display(nfi_uploader)

    # ipywidgets 7 returns a dict {name: info}; ipywidgets 8 returns a tuple of dicts.
    uploads = nfi_uploader.value
    if hasattr(uploads, 'items'):
        uploads_iterable = list(uploads.items())
    else:
        uploads_iterable = [(u['name'], u) for u in uploads]

    for name, file_info in uploads_iterable:
        content = io.BytesIO(file_info['content'])
        if name.endswith(('.csv', '.xlsx', '.xls')):
            df = pd.read_csv(content) if name.endswith('.csv') else pd.read_excel(content)
            print(f"✅ SUCCESS: Loaded Tabular File -> {name}")
            display(df.head())
            print(f"📏 Data Shape: {df.shape[0]} rows, {df.shape[1]} columns")
        elif name.endswith(('.gpkg', '.geojson')):
            tmp_path = os.path.join('/tmp', os.path.basename(name))
            with open(tmp_path, 'wb') as f:
                f.write(file_info['content'])
            gdf = gpd.read_file(tmp_path)
            print(f"✅ SUCCESS: Loaded Spatial File -> {name}")
            fig, ax = plt.subplots(figsize=(6, 6))
            gdf.plot(ax=ax, cmap='Greens')
            plt.title("NFI Plot Points Preview")
            display(fig)

    display(HTML("<h3 style='color:green;'>✅ Uploads Validated! Proceed to Notebook 01.</h3>"))


nfi_uploader.observe(on_upload_change, names='value')